In [1]:
import os
import fiftyone as fo
import fiftyone.zoo as foz
import optuna
import pandas as pd
from ultralytics import YOLO
from pathlib import Path
import yaml
from PIL import Image
import matplotlib.pyplot as plt
import random
from dotenv import load_dotenv

/home/mikel/github/TKNIKA/kortxovision/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(override=True)


True

In [3]:
fo.config.dataset_zoo_dir = os.getenv("DATA_FOLDER")
train_dataset_name = os.getenv("TRAIN_DATASET_NAME")
val_dataset_name = os.getenv("VAL_DATASET_NAME")
test_dataset_name = os.getenv("TEST_DATASET_NAME")
yolo_train_folder = os.getenv("YOLO_TRAIN_FOLDER")
yolo_best_model_path = os.getenv("YOLO_BEST_MODEL_PATH")
datasets = [train_dataset_name, val_dataset_name, test_dataset_name]
print(datasets)

['train_1500', 'val_1500', 'test_1500']


In [4]:
import optuna
from ultralytics import YOLO

def objective(trial):
    epochs = trial.suggest_int('epochs', 50, 150)
    lr0 = trial.suggest_float('lr0', 1e-4, 1e-2, log=True)
    lrf = trial.suggest_float('lrf', 0.01, 0.5)
    
    model = YOLO('yolov8n.pt')
    
    results = model.train(
        data=yolo_train_folder,
        epochs=epochs,
        imgsz=640,
        project='yolo/runs/optuna',
        lr0=lr0,
        lrf=lrf,
        name=f'trial_{trial.number}',
        verbose=False
    )
    
    # Retorna la métrica que quieres optimizar
    return results.results_dict['metrics/mAP50-95(B)']

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print('Best params:', study.best_params)
print('Best mAP50-95:', study.best_value)

[I 2025-09-29 11:09:32,052] A new study created in memory with name: no-name-64f6a4e5-d94f-498b-bfd3-c69ab808922f


Ultralytics 8.3.203 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6141MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/mikel/github/TKNIKA/kortxovision/data/yolo/train/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=90, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00034215894563355655, lrf=0.454332504758712, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=trial_0, nbs=64, nms=False, opset=None, opti

E0000 00:00:1759136975.138372    3453 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759136975.163387    3453 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1759136975.333111    3453 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1759136975.333181    3453 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1759136975.333182    3453 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1759136975.333183    3453 computation_placer.cc:177] computation placer already registered. Please check linka

TensorBoard: Start with 'tensorboard --logdir /home/mikel/github/TKNIKA/kortxovision/yolo/runs/optuna/trial_0', view at http://localhost:6006/
Overriding model.yaml nc=80 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                 

[W 2025-09-29 11:33:35,169] Trial 0 failed with parameters: {'epochs': 90, 'lr0': 0.00034215894563355655, 'lrf': 0.454332504758712} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/mikel/github/TKNIKA/kortxovision/.venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_3453/2675252818.py", line 11, in objective
    results = model.train(
              ^^^^^^^^^^^^
  File "/home/mikel/github/TKNIKA/kortxovision/.venv/lib/python3.12/site-packages/ultralytics/engine/model.py", line 800, in train
    self.trainer.train()
  File "/home/mikel/github/TKNIKA/kortxovision/.venv/lib/python3.12/site-packages/ultralytics/engine/trainer.py", line 235, in train
    self._do_train()
  File "/home/mikel/github/TKNIKA/kortxovision/.venv/lib/python3.12/site-packages/ultralytics/engine/trainer.py", line 436, in _do_train
    self.op

KeyboardInterrupt: 